[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-priceelasticity.ipynb)

# Full Project: Retail Price Optimization (Price Elasticity of Demand)

*AIBits Academy · Machine Learning End To End · Full Project*

A quick-service café chain's burger pricing, modelled with OLS regression — and a revenue-maximizing price that isn't the current one.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://raw.githubusercontent.com/dasari-mohana-zz/Cafe_Price_Optimization_project/main/Cafe_Sell_Meta_Data.csv', 'Cafe_Sell_Meta_Data.csv')
fetch('https://raw.githubusercontent.com/dasari-mohana-zz/Cafe_Price_Optimization_project/main/Cafe_Transaction_Store.csv', 'Cafe_Transaction_Store.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A quick-service café chain — the same pricing question faced by any Indian QSR outlet in Surat or Ahmedabad selling snacks and beverages — sells burgers, coke, coffee, and lemonade, sometimes standalone and sometimes as combos. Setting prices too high loses sales; setting them too low leaves margin on the table. The goal is to find, for each item/deal, the price that **maximizes revenue** — not just the price customers happen to be paying today.

> **Dataset**
>
> **5,404 daily sales-transaction records across 2012**, covering 4 items (burger, coke, coffee, lemonade) sold under 11 distinct "deals" (standalone or combo), plus a calendar dimension table. [Transactions →](https://raw.githubusercontent.com/dasari-mohana-zz/Cafe_Price_Optimization_project/main/Cafe_Transaction_Store.csv) · [Product metadata →](https://raw.githubusercontent.com/dasari-mohana-zz/Cafe_Price_Optimization_project/main/Cafe_Sell_Meta_Data.csv) · [Date info →](https://raw.githubusercontent.com/dasari-mohana-zz/Cafe_Price_Optimization_project/main/Cafe_DateInfo.csv)

## Step 1 — Load and Merge

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols

sold = pd.read_csv('Cafe_Sell_Meta_Data.csv')
trans = pd.read_csv('Cafe_Transaction_Store.csv')

data = trans.merge(sold, on=['SELL_ID','SELL_CATEGORY'])
print(data['ITEM_NAME'].value_counts())

Burger appears in every transaction because it's the anchor item of every deal — sold alone (SELL_ID 1070) or bundled with coke (2051), lemonade (2052), or coke+coffee (2053). Coke, lemonade, and coffee never appear as standalone purchases in this data — they only ever sell as part of a burger combo, so pricing decisions for those items really mean pricing the *combo*, not the individual drink.

## Step 2 — Price Elasticity per Deal

For each of the four burger-anchored deals, fit **Quantity ~ Price** by ordinary least squares, then compute elasticity as the regression slope scaled by the ratio of average price to average quantity:

$$\text{Elasticity } (\varepsilon) = \text{slope} \times \dfrac{\text{Average Price}}{\text{Average Quantity}}$$

In [ ]:
deals = [('BURGER',1070), ('BURGER',2051), ('BURGER',2052), ('BURGER',2053)]
for item, sid in deals:
    d = data[(data.ITEM_NAME==item)&(data.SELL_ID==sid)]
    model = ols("QUANTITY ~ PRICE", data=d).fit()
    mean_p, mean_q = d.PRICE.mean(), d.QUANTITY.mean()
    elasticity = model.params['PRICE'] * (mean_p/mean_q)
    print(f"SELL_ID {sid}: avg price=₹{mean_p:.2f}, elasticity={elasticity:.3f}, R²={model.rsquared:.3f}")

All four deals show **elastic demand** (|ε| generally above or near 1) — a 1% price increase costs roughly 0.86% to 1.36% of volume, depending on the deal. The triple-combo (2053) has both the strongest elasticity and the best model fit (R²=0.247), suggesting price is a comparatively more dominant driver of its volume than for the simpler deals, where day-of-week, weather, and holiday effects (available in the date-info table but not modelled here) likely explain more of the remaining variance.

## Step 3 — Finding the Revenue-Maximizing Price

For the standalone burger (SELL_ID 1070), revenue as a function of price is Price × Quantity. Substituting the fitted linear demand curve Quantity = a + b·Price gives a quadratic revenue function, maximized where its derivative is zero:

$$\begin{gathered}\mathrm{Revenue}(P) = P\cdot(a+bP) = aP + bP^2 \\[6pt] \dfrac{d(\mathrm{Revenue})}{dP} = a + 2bP = 0 \;\;\implies\;\; P^* = \dfrac{-a}{2b}\end{gathered}$$

In [ ]:
burger = data[(data.ITEM_NAME=='BURGER')&(data.SELL_ID==1070)]
model = ols("QUANTITY ~ PRICE", data=burger).fit()
a, b = model.params['Intercept'], model.params['PRICE']

optimal_price = -a / (2*b)
optimal_qty = a + b*optimal_price
current_price, current_qty = burger.PRICE.mean(), burger.QUANTITY.mean()

print(f"Model: Qty = {a:.2f} + ({b:.3f})·Price   [R²={model.rsquared:.3f}]")
print(f"Current: price=₹{current_price:.2f}, qty/day={current_qty:.1f}, revenue/day=₹{current_price*current_qty:.2f}")
print(f"Optimal: price=₹{optimal_price:.2f}, qty/day={optimal_qty:.1f}, revenue/day=₹{optimal_price*optimal_qty:.2f}")

The revenue-maximizing price (₹13.28) is **lower** than the current average price (₹15.16) — the model predicts that dropping the price would sell enough extra volume (81.4 → 94.8 units/day) to more than offset the lower per-unit margin, for a **+2.1% revenue lift**. This is a smaller, more realistic uplift than a naive "just lower the price and see" experiment might suggest, precisely because it accounts for how much extra volume a given price cut actually buys, rather than assuming demand is infinitely elastic.

## Visualizing the Revenue Curve

The curve is Revenue(P) = P×(189.68 − 7.141×P), plotted directly from the fitted model. Current price sits just past the peak — the optimal price is a small step down, not a large one.

> **💡 R²=0.105 is low — should this recommendation be trusted?**
>
> An R² this low means price alone explains only about 10% of the day-to-day variation in quantity sold — the rest is likely weekday/weekend effects, school-break seasonality, and temperature (all present in the date-info table but not used in this simplified model). The *direction* of the elasticity estimate is still usable for a pricing decision, but before rolling out a price change across every outlet, a business would want to run it as a controlled experiment on a subset of stores first — exactly the kind of design question the A/B Testing full project's Minimum Detectable Effect framework addresses.

## Key Business Takeaways

- All four burger-anchored deals show elastic demand (|elasticity| 0.86–1.36) — customers are meaningfully price-sensitive across every pricing configuration tested, not just the premium standalone item.
- The revenue-maximizing price for the standalone burger (₹13.28) is below the current average (₹15.16) — a +2.1% predicted revenue lift, driven by extra volume more than offsetting the lower margin per unit.
- Low R² (0.10–0.25 across deals) is itself a business signal: day-of-week, holidays, and weather likely matter more than the model currently captures — a reason to validate any pricing change with a controlled test before a full rollout.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Average price per deal

Store in `avg_price` the mean `PRICE` of the burger rows for each `SELL_ID` (a Series indexed by SELL_ID).

In [ ]:
avg_price = None   # TODO


In [ ]:
try:
    check("standalone burger about 15.16", abs(avg_price[1070] - 15.16) < 0.01)
    check("four deals", len(avg_price) == 4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
avg_price = data[data.ITEM_NAME == "BURGER"].groupby("SELL_ID")["PRICE"].mean()

```

</details>

### Exercise 2 · Medium · Elasticity as a function

Write `elasticity(d)` for a frame of one deal: fit `QUANTITY ~ PRICE` with `ols` and return `slope * mean_price / mean_quantity`.

In [ ]:
def elasticity(d):
    pass   # TODO


In [ ]:
try:
    d1070 = data[(data.ITEM_NAME == "BURGER") & (data.SELL_ID == 1070)]
    check("standalone burger = -1.330", round(elasticity(d1070), 3) == -1.33)
    d2051 = data[(data.ITEM_NAME == "BURGER") & (data.SELL_ID == 2051)]
    check("burger + coke = -0.860", round(elasticity(d2051), 3) == -0.86)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def elasticity(d):
    m = ols("QUANTITY ~ PRICE", data=d).fit()
    return m.params["PRICE"] * d.PRICE.mean() / d.QUANTITY.mean()

```

</details>

### Exercise 3 · Stretch · Best price for the triple combo

For the burger + coke + coffee deal (`SELL_ID` 2053) fit the linear demand curve `Qty = a + b·Price`, and store the revenue-maximising price `-a / (2b)` in `p_star`. Set `better_than_now` = whether the fitted revenue at `p_star` beats the fitted revenue at the current average price.

In [ ]:
p_star = better_than_now = None   # TODO


In [ ]:
try:
    d = data[(data.ITEM_NAME == "BURGER") & (data.SELL_ID == 2053)]
    m = ols("QUANTITY ~ PRICE", data=d).fit()
    a, b = m.params["Intercept"], m.params["PRICE"]
    rev = lambda p: p * (a + b * p)
    check("price is positive", p_star > 0)
    check("optimum beats the current price", better_than_now is True and rev(p_star) >= rev(d.PRICE.mean()))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
d = data[(data.ITEM_NAME == "BURGER") & (data.SELL_ID == 2053)]
m = ols("QUANTITY ~ PRICE", data=d).fit()
a, b = m.params["Intercept"], m.params["PRICE"]
p_star = -a / (2 * b)
rev = lambda p: p * (a + b * p)
better_than_now = bool(rev(p_star) >= rev(d.PRICE.mean()))

```

A linear demand curve gives a quadratic revenue function; its peak is where marginal revenue hits zero. Always sanity-check the fit (R² is low here) before repricing.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Retail Price Optimization (Price Elasticity of Demand)**.*